---


# **Part 0: Overview**

---

# **A quick Recap, This Notebook Runs in a few Parts:**

 **Part I: boltz-2 prediction:**
1. Takes a **`UniProt accession ID`** as input.

2. Checks with **UniProt REST API** that:
   * `entryType` is **"UniProtKB reviewed (Swiss-Prot)"**
   * `organism.scientificName` contains **"Homo sapiens"**

3. If valid:
   * Fetches the `protein sequence` from **UniProt**.
   * Allows you to parameterize ligands for **Boltz-2 structure prediction**
   * Let's you run a **Boltz-2 structure prediction** and save returned **mmCIF** as **`UniProt-ID`.pdb**

 **Part II: allele frequency mapping to.pdb:**

1. Retrieves formatted **`gnomAD allele frequency`** data for **`UniProt accession ID`**

2. Allows upload of pdb that should match **`UniProt accession ID`** **(no sequence gaps)**

3. You can specify **`allele frequency`** or **`ancestry frequency`** and build models for later visualization
4. obtain a special **`UniProt accession ID`_af_gnomad.pdb** or **_category_gnomad.pdb** for visualizing minor allele hotspots and/or sidechains

 **Part III: Tutorial/walkthrough for visualizing _gnomad.pdb files.**

1.   List item
2.   List item





---


# **Part I: boltz-2 prediction**

---



#**⚠️ Before you start: enable a GPU runtime (Boltz-2 requires a GPU to run).**

In Google Colab:
1.   Click Runtime → Change runtime type
2.   Set Hardware accelerator to GPU (T4 recommended)
3.   **Click Save**







#**🤖 Let’s start by installing the required environment for Boltz-2**


In [ ]:
# @title Automated Boltz-2 Dependencies Install & Restart
import subprocess
import sys
import os
from IPython.display import display, HTML

# --- 1. Define Helper Function for Install/Uninstall ---
def run_command(cmd):
    """Run a shell command and print the output."""
    print(f"$ {cmd}")
    try:
        subprocess.run(cmd, shell=True, check=True, text=True, capture_output=False)
    except subprocess.CalledProcessError as e:
        print(f"⚠️ Command failed (might be expected for uninstalls): {e}")

# --- 2. Uninstall potential conflicting versions ---
print("--- 1/4: Removing old dependencies... ---")
# Uninstall old versions to ensure clean install
run_command("pip uninstall -y numpy scipy numba || true")

# --- 3. Install specific, known-compatible versions ---
print("\n--- 2/4: Installing compatible dependencies... ---")
# Pinning versions that work well with Numba and Boltz-2
run_command("pip install -q 'numpy<=2.2.0' 'scipy<2.3.0' numba")

# --- 4. Install Boltz-2 with CUDA support ---
print("\n--- 3/4: Installing boltz[cuda]... ---")
run_command("pip install -q -U 'boltz[cuda]'")

# --- 5. AUTOMATIC RESTART COMMAND (Colab specific) ---
print("\n--- 4/4: Triggering Automatic Runtime Restart... ---")
print("🚨 The notebook will now restart to load the fixed dependencies. Please wait...")

# Colab-specific command to automatically restart the Python kernel
# Note: You MUST run the next code cell (Boltz-2 usage) AFTER the restart.
display(HTML("<script>Jupyter.notebook.session.restart({kernel_name: 'python3'})</script>"))

# Add a protective print statement in case the restart command fails
print("\n⚠️ If the notebook does not restart in 10 seconds, please manually go to Runtime -> Restart runtime.")

--- 1/4: Removing old dependencies... ---
$ pip uninstall -y numpy scipy numba || true

--- 2/4: Installing compatible dependencies... ---
$ pip install -q 'numpy<=2.2.0' 'scipy<2.3.0' numba


KeyboardInterrupt: 

# **✅  Let's validate UniProt ID (human reference) and retrieve sequence**

In [ ]:
#@title Input UniProt ID, then run this cell
#@markdown This cell validates the UniProt entry and fetches the protein sequence.
#@markdown **Criteria (must pass):**
#@markdown - `UniProtKB reviewed (Swiss-Prot)`
#@markdown - `Homo sapiens (Human)`

uniprot_id = "Q9Y2D9"  #@param {type:"string"}

import requests

UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb/{accession}?format=json"

# Globals for downstream steps
current_uniprot_id = None
current_sequence = None

def fetch_uniprot_entry(accession: str) -> dict:
    accession = accession.strip()
    if not accession:
        raise ValueError("UniProt ID is empty.")
    url = UNIPROT_ENTRY_URL.format(accession=accession)
    r = requests.get(url)
    if r.status_code == 404:
        raise ValueError(f"No UniProtKB entry found for '{accession}'.")
    if not r.ok:
        raise ValueError(f"UniProt API error (HTTP {r.status_code}).")
    return r.json()

def validate_entry(entry: dict) -> tuple[bool, str]:
    entry_type = entry.get("entryType", "")
    sci_name = entry.get("organism", {}).get("scientificName", "")

    is_reviewed = entry_type == "UniProtKB reviewed (Swiss-Prot)"
    is_human = "Homo sapiens" in sci_name

    lines = []
    lines.append("✓ UniProtKB reviewed (Swiss-Prot)" if is_reviewed else f"✗ entryType: {entry_type}")
    lines.append("✓ Organism: Homo sapiens (Human)" if is_human else f"✗ Organism: {sci_name}")
    return (is_reviewed and is_human), "\n".join(lines)

def extract_sequence(entry: dict) -> str:
    seq = entry.get("sequence", {}).get("value")
    if not seq:
        raise ValueError("No sequence found in UniProt entry.")
    return seq.replace("\n", "").strip()

# ---- run ----
try:
    entry = fetch_uniprot_entry(uniprot_id)
    ok, msg = validate_entry(entry)

    print(f"Checking UniProt entry for {uniprot_id}\n")
    print(msg)
    print()

    if not ok:
        print("❌ Rejected: entry does not meet criteria.")
    else:
        seq = extract_sequence(entry)
        current_uniprot_id = uniprot_id.strip()
        current_sequence = seq
        print(f"✅ Accepted. Sequence length: {len(seq)} aa")
        # Optional: preview first 60 aa
        print("Sequence preview:", seq[:60] + ("..." if len(seq) > 60 else ""))

except Exception as e:
    print("❌ Error:", e)


# **🔧 Let's parameterize a Boltz-2 job**

In [ ]:
#@title Build Boltz-2 YAML (protein + ligands + constraints)
#@markdown Uses `current_sequence` from the UniProt step.
#@markdown - CCD ligands:   entries separated by ':' or ';' (optional explicit IDs via ID@CCD)
#@markdown - SMILES ligands: entries separated by ':' or ';' (NO explicit IDs; auto-assigned chain IDs)
#@markdown - Bond constraints (can provide multiple, separated by ':' or newlines):
#@markdown     CHAIN,RES,ATOM;CHAIN,RES,ATOM
#@markdown   Example:
#@markdown     A,461,SG;H,1,FE : A,462,SG;H,1,FE

jobname = "job1"  #@param {type:"string"}
protein_chain_id = "A"     #@param {type:"string"}

ligand_ccd = ""        #@param {type:"string"}
ligand_smiles = ""     #@param {type:"string"}

bond_constraints = ""      #@param {type:"string"}

print("Expected semantics:")
print("  - Protein: one entry with id=protein_chain_id and sequence=current_sequence")
print("  - Ligands:")
print("      * CCD   (ligand_ccd): 'HEM:HOH' or 'H@HEM;W@HOH' (':' or ';' separators)")
print("      * SMILES(ligand_smiles): 'CCO:CCN' or 'CCO;CCN'  (NO 'ID@...' allowed)")
print("        SMILES chains auto-assign after the protein chain (e.g., B, C, D...)")
print("  - Bond constraints (bond_constraints):")
print("      * One or more entries separated by ':' or newlines")
print("      * Each entry: CHAIN,RES,ATOM;CHAIN,RES,ATOM")
print("        Example: A,461,SG;H,1,FE")
print()

import re
import string
from pathlib import Path
import yaml

# --- YAML helpers to force inline [A, 461, SG] style ---
class FlowSeq(list):
    """A list that should be emitted in YAML flow style: [a, b, c]."""
    pass

class IndentDumper(yaml.SafeDumper):
    def increase_indent(self, flow=False, indentless=False):
        return super().increase_indent(flow, False)

def represent_flowseq(dumper, data):
    return dumper.represent_sequence("tag:yaml.org,2002:seq", data, flow_style=True)

IndentDumper.add_representer(FlowSeq, represent_flowseq)

# --- parsing helpers ---
def split_multi_sep(s: str):
    """Split on ':' or ';' for ligand fields."""
    s = (s or "").strip()
    if not s:
        return []
    return [x.strip() for x in re.split(r"[:;]", s) if x.strip()]

def split_bond_entries(s: str):
    """
    Split multiple bond constraints on ':' or newlines.
    NOTE: ';' is reserved inside each bond entry (atom1 ; atom2).
    """
    s = (s or "").strip()
    if not s:
        return []
    parts = []
    for line in s.splitlines():
        line = line.strip()
        if not line:
            continue
        parts.extend([x.strip() for x in line.split(":") if x.strip()])
    return parts

def parse_one_bond(entry: str):
    """
    Parse one bond entry: CHAIN,RES,ATOM;CHAIN,RES,ATOM
    Emits exactly:
      {"bond": {"atom1": [A, 461, SG], "atom2": [H, 1, FE]}}
    with atom1/atom2 lists in flow style.
    """
    entry = (entry or "").strip()
    if not entry:
        return None
    if ";" not in entry:
        raise ValueError("Each bond entry must use ';' between the two atoms: CHAIN,RES,ATOM;CHAIN,RES,ATOM")

    left, right = [x.strip() for x in entry.split(";", 1)]
    a1 = [x.strip() for x in left.split(",")]
    a2 = [x.strip() for x in right.split(",")]

    if len(a1) != 3 or len(a2) != 3:
        raise ValueError("Bond entry must be CHAIN,RES,ATOM;CHAIN,RES,ATOM (3 comma-separated fields each side)")

    try:
        res1 = int(a1[1])
        res2 = int(a2[1])
    except ValueError:
        raise ValueError("Bond entry residue indices (RES) must be integers.")

    return {
        "bond": {
            "atom1": FlowSeq([a1[0], res1, a1[2]]),
            "atom2": FlowSeq([a2[0], res2, a2[2]]),
        }
    }

def parse_ccd_with_optional_ids(raw: str):
    items = split_multi_sep(raw)
    out = []
    for it in items:
        if "@" in it:
            cid, val = it.split("@", 1)
            cid = cid.strip()
            val = val.strip()
            if not cid or not val:
                raise ValueError(f"Invalid CCD entry '{it}'. Use ID@CCD or CCD.")
            out.append((cid, val, True))
        else:
            out.append((None, it, False))
    return out

def parse_smiles_no_ids(raw: str):
    items = split_multi_sep(raw)
    for it in items:
        if "@" in it:
            raise ValueError(
                "SMILES input must NOT include explicit chain IDs. "
                "Remove 'ID@...' and provide only SMILES separated by ':' or ';'."
            )
    return items

def make_auto_chain_generator(start_after: str):
    letters = list(string.ascii_uppercase)
    start_idx = letters.index(start_after) + 1 if start_after in letters else 1
    return iter(letters[start_idx:])

# Require UniProt sequence from previous step
if "current_sequence" not in globals() or current_sequence is None:
    raise RuntimeError("Run the UniProt validation cell first (so current_sequence exists).")

seq_entries = [{"protein": {"id": protein_chain_id, "sequence": current_sequence}}]
auto_chain_iter = make_auto_chain_generator(protein_chain_id)

# CCD ligands (optional explicit IDs)
for cid, ccd, has_id in parse_ccd_with_optional_ids(ligand_ccd):
    chain = cid if has_id else next(auto_chain_iter)
    seq_entries.append({"ligand": {"id": chain, "ccd": ccd.upper()}})

# SMILES ligands (NO explicit IDs; always auto chain)
for smi in parse_smiles_no_ids(ligand_smiles):
    chain = next(auto_chain_iter)
    seq_entries.append({"ligand": {"id": chain, "smiles": smi}})

# Bond constraints (multiple entries allowed via ':' or newlines)
constraints = []
for entry in split_bond_entries(bond_constraints):
    b = parse_one_bond(entry)
    if b:
        constraints.append(b)

config = {"version": 1, "sequences": seq_entries}
if constraints:
    config["constraints"] = constraints

out_dir = Path("boltz_inputs")
out_dir.mkdir(exist_ok=True)
yaml_path = out_dir / f"{jobname}.yaml"

yaml_text = yaml.dump(
    config,
    Dumper=IndentDumper,
    sort_keys=False,
    default_flow_style=False,
    indent=2,
    width=10_000,
)

yaml_path.write_text(yaml_text)

print("✅ Wrote Boltz-2 YAML to:", yaml_path)
print("\n--- YAML preview ---\n")
print(yaml_text)


# **📥 Download generated Boltz-2 YAML (Optional)**




In [ ]:
#@title View the generated YAML (notepad-style)
#@markdown Shows and download the most recent YAML in `boltz_inputs/` in a scrollable "notepad" box.

from pathlib import Path
from IPython.display import display, HTML
import html
from google.colab import files

files.download(str(yaml_path))

yaml_dir = Path("boltz_inputs")
yaml_files = sorted(yaml_dir.glob("*.yaml"), key=lambda p: p.stat().st_mtime)

if not yaml_files:
    raise RuntimeError("No YAML files found in boltz_inputs/. Run the YAML generation cell first.")

yaml_path = yaml_files[-1]
yaml_text = yaml_path.read_text()

# escape for HTML display
safe_text = html.escape(yaml_text)

display(HTML(f"""
<div style="
  border: 1px solid #ddd;
  border-radius: 8px;
  padding: 12px;
  background: #fafafa;
  font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', monospace;
  font-size: 13px;
  line-height: 1.4;
  white-space: pre;
  overflow: auto;
  max-height: 420px;
">
<b>{yaml_path.name}</b>\n
{safe_text}
</div>
"""))


#**🚀  Run Boltz-2**

In [ ]:
#@title Run Boltz-2 prediction on most recently generated YAML (using MSA server). This can take a little while ... Stretch break 🤸‍♂️ 🧘‍♀️



import subprocess
from pathlib import Path

# Locate latest YAML
yaml_dir = Path("boltz_inputs")
yaml_files = sorted(yaml_dir.glob("*.yaml"), key=lambda p: p.stat().st_mtime)

if not yaml_files:
    raise RuntimeError("No YAML files found in boltz_inputs/. Generate a YAML first.")

yaml_path = yaml_files[-1]
out_dir = Path("boltz_outputs")
out_dir.mkdir(exist_ok=True)

cmd = [
    "boltz", "predict",
    str(yaml_path),
    "--use_msa_server",
    "--out_dir", str(out_dir),
]

print("🚀 Running Boltz-2 with MSA server")
print("YAML:", yaml_path.name)
print("Output dir:", out_dir)
print("\nCommand:")
print(" ".join(cmd))
print("\n---\n")

# Capture stdout and stderr for better error diagnosis
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("\n❌ Boltz-2 prediction failed.")
    print("--- Standard Output ---")
    print(result.stdout)
    print("--- Standard Error ---")
    print(result.stderr)
    raise RuntimeError("Boltz-2 prediction failed. Check logs above for details.")

print("\n✅ Boltz-2 prediction complete.")
# Show output directory structure (correct Boltz layout)
job_out_dir = out_dir / f"boltz_results_{yaml_path.stem}"

if not job_out_dir.exists():
    raise FileNotFoundError(f"Expected output directory not found: {job_out_dir}")

print("\n📁 Output directory structure:")
print(job_out_dir.name + "/")

for path in sorted(job_out_dir.rglob("*")):
    level = len(path.relative_to(job_out_dir).parts)
    indent = "  " * level
    suffix = "/" if path.is_dir() else ""
    print(f"{indent}- {path.name}{suffix}")


#**🔬 View prediction**

In [ ]:
#@title py3Dmol protein rendering (rainbow cartoon; HETATM as sticks.)


!pip -q install biopython py3Dmol

import py3Dmol
from pathlib import Path
from Bio.PDB import MMCIFParser, PDBIO

cif_path = Path("boltz_outputs/boltz_results_job1/predictions/job1/job1_model_0.cif")
pdb_path = cif_path.with_suffix(".pdb")

assert cif_path.exists(), f"Missing CIF file: {cif_path}"

# CIF → PDB
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("boltz_model", str(cif_path))

io = PDBIO()
io.set_structure(structure)
io.save(str(pdb_path))   # ← important fix

pdb_text = pdb_path.read_text()

view = py3Dmol.view(width=900, height=550)
view.addModel(pdb_text, "pdb")

# Clear any default styles
view.setStyle({}, {})

# Protein (ATOM) → rainbow cartoon
view.setStyle(
    {"hetflag": False},
    {"cartoon": {"color": "spectrum"}}
)

# HETATM → sticks
view.setStyle(
    {"hetflag": True},
    {"stick": {"radius": 0.25}}
)

view.zoomTo()
view.show()


#**📥 Download boltz-2 prediction as `UniProt-ID.pdb`**

In [ ]:
#@title Download .pdb

from pathlib import Path
import shutil

# pick the UniProt id you set earlier
uid = (globals().get("current_uniprot_id") or globals().get("uniprot_id") or "uniprotid").strip()

# this should already exist from your CIF→PDB step
src = globals().get("pdb_path", None)
if src is None:
    raise ValueError("pdb_path not found. Make sure you ran the CIF→PDB conversion cell first.")
src = Path(src)

if not src.exists():
    raise FileNotFoundError(f"Missing PDB file: {src}")

dst = Path(f"{uid}.pdb")
shutil.copyfile(src, dst)
print(f"✅ Saved as: {dst.resolve()}")

# Download (Colab) or show a link (Jupyter)
try:
    from google.colab import files  # type: ignore
    files.download(str(dst))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(dst)))




---


# **Part II: allele frequency mapping to .pdb |**
**MORPH (Map Missense on Reference Protein Human)**


---



#**⚠️ Before you continue: change runtime from GPU to CPU (no need to waste ressources 🔥💸 or 🌳)**

In Google Colab:

1.   Click Runtime → Change runtime type
2.   Set Hardware accelerator to CPU
3.   **Click Save**

#**⚠️ to map allele frequency to .pdb 🧭🗺️📍**

Requires a pdb meeting at least these **`requirements`**:

1.   whose **`chain A`** sequence matches **`UniProtKB reviewed (Swiss-Prot)`** & **`Homo sapiens (Human)`** entry
2.   has **contiguous sequence** (**i.e no gaps**)
3.   has available corresponding **`gnomAD allele frequency`** data




#**🧱 Let’s start by installing the required environment for MORPH**


In [ ]:
# @title 🧱 Automated MORPH Dependencies Install & Restart (session will crash!)
import os, subprocess, requests
from pathlib import Path

def run(cmd: str, check: bool = True, show: bool = True):
    """Run shell command. If show=False, hide stdout unless error."""
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if show and p.stdout.strip():
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr.strip():
            print(p.stderr)
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = os.environ.get(name)
    if not val:
        raise RuntimeError(f"Missing secret '{name}' (Colab → Secrets).")
    return val

# -----------------------------
# Config
# -----------------------------
PYMUT_REPO = "https://github.com/gerdos/pyMUT"

OWNER = "lucaschestnuttree"
PRIVATE_REPO = "missense_structure_view"
REF = "main"

FILES = [
    "src/2-mapping/map_2_structure/pymut_gnomad_af_2_structure.py",
    "src/2-mapping/map_2_structure/pymut_gnomad_anc_2_structure.py",
]

DEST_DIR = Path("pyMUT/src/2-mapping/map_2_structure")

NUMPY_VER  = "1.26.4"
PANDAS_VER = "2.1.4"

# -----------------------------
# 0) Read PAT safely
# -----------------------------
GITHUB_PAT = get_secret("Github_PAT")

# -----------------------------
# 1) Install deps (quiet-ish)
# -----------------------------
print("1) Cleaning packages...")
run("python -m pip uninstall -y numpy pandas biopython || true", check=False, show=False)
run("python -m pip cache purge || true", check=False, show=False)

print(f"2) Installing numpy=={NUMPY_VER}, pandas=={PANDAS_VER}, biopython...")
run("python -m pip install -U pip", show=False)

# Pin numpy first
run(f"python -m pip install --no-cache-dir --upgrade --force-reinstall 'numpy=={NUMPY_VER}'", show=False)

# Install pandas/biopython without deps so they can't upgrade numpy
run(f"python -m pip install --no-cache-dir --upgrade --force-reinstall --no-deps 'pandas=={PANDAS_VER}'", show=False)
run("python -m pip install --no-cache-dir --upgrade --force-reinstall --no-deps 'biopython>=1.7'", show=False)

# Re-assert numpy pin at end
run(f"python -m pip install --no-cache-dir --upgrade --force-reinstall 'numpy=={NUMPY_VER}'", show=False)

# Print only the essentials
print("3) Sanity check:")
run(
    "python - << 'PY'\n"
    "import sys, numpy, pandas, Bio\n"
    "print('Python:', sys.executable)\n"
    "print('numpy:', numpy.__version__)\n"
    "print('pandas:', pandas.__version__)\n"
    "print('biopython:', Bio.__version__)\n"
    "PY",
    show=True
)

# -----------------------------
# 2) Clone public pyMUT
# -----------------------------
print("4) Ensuring pyMUT repo exists...")
if not Path("pyMUT").exists():
    run(f"git clone --depth 1 {PYMUT_REPO}", show=False)
    print("   ✅ cloned pyMUT")
else:
    print("   ✅ pyMUT already exists")

# -----------------------------
# 3) Fetch private scripts via GitHub API
# -----------------------------
print("5) Downloading private scripts...")
DEST_DIR.mkdir(parents=True, exist_ok=True)

sess = requests.Session()
sess.headers.update({
    "Authorization": f"Bearer {GITHUB_PAT}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "colab-pymut-setup",
})

def download_from_private_repo(repo_path: str):
    api_url = f"https://api.github.com/repos/{OWNER}/{PRIVATE_REPO}/contents/{repo_path}"
    r = sess.get(api_url, params={"ref": REF}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"GitHub API error for '{repo_path}': HTTP {r.status_code}\n{r.text[:400]}")
    data = r.json()
    download_url = data.get("download_url")
    if not download_url:
        raise RuntimeError(f"No download_url returned for '{repo_path}'")
    r2 = sess.get(download_url, timeout=60)
    if r2.status_code != 200:
        raise RuntimeError(f"Download error for '{repo_path}': HTTP {r2.status_code}\n{r2.text[:400]}")
    out_path = DEST_DIR / Path(repo_path).name
    out_path.write_bytes(r2.content)
    print(f"   ✅ wrote {out_path}")

for pth in FILES:
    download_from_private_repo(pth)

print("\n💥 Forcing runtime restart to reload NumPy...")
os.kill(os.getpid(), 9)


1) Cleaning packages...
2) Installing numpy==1.26.4, pandas==2.1.4, biopython...
3) Sanity check:
Python: /usr/bin/python3
numpy: 1.26.4
pandas: 2.1.4
biopython: 1.87

4) Ensuring pyMUT repo exists...
   ✅ cloned pyMUT
5) Downloading private scripts...
   ✅ wrote pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_af_2_structure.py


In [ ]:
# @title 🔌 After restart: make pyMUT importable
import os, sys
from pathlib import Path

PYMUT_DIR = Path("pyMUT").resolve()

# Notebook imports
if str(PYMUT_DIR) not in sys.path:
    sys.path.insert(0, str(PYMUT_DIR))

# Subprocess imports
os.environ["PYTHONPATH"] = str(PYMUT_DIR) + (":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")

import numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("PYTHONPATH:", os.environ["PYTHONPATH"])

import pymut
print("✅ pymut imported from:", pymut.__file__)


Python: /usr/bin/python3
NumPy: 1.26.4
PYTHONPATH: /content/pyMUT:/env/python
✅ pymut imported from: /content/pyMUT/pymut.py


In [ ]:
#@title #🔍 Retrieve necessary lookup table
# =======================
# Zenodo record downloader (token extracted from ZENODO_SECRET_LINK)
# =======================

import os
import time
import requests
import re

# ---- CONFIG ----
RECORD_ID = "18163874"
FILENAME = "lookup_amissenseXgnomAD.tsv"
OUT_DIR = "zenodo_downloads"

# ---- TOKEN HELPERS ----
def get_zenodo_token_from_secret() -> str:
    """
    Reads ZENODO_SECRET_LINK from:
      1) Colab Secrets (google.colab.userdata)
      2) Environment variable

    Accepts either:
      - full link containing token=...
      - raw token string
    """
    secret = None
    try:
        from google.colab import userdata
        secret = userdata.get("ZENODO_SECRET_LINK")
    except Exception:
        secret = os.environ.get("ZENODO_SECRET_LINK")

    if not secret:
        raise ValueError("ZENODO_SECRET_LINK not found in Colab Secrets (or env).")

    secret = secret.strip()

    # If a full link (old approach)
    if "token=" in secret:
        return secret.split("token=", 1)[1].split("&", 1)[0]

    # If it's already a raw token
    if re.fullmatch(r"[A-Za-z0-9\-_\.]+", secret):
        return secret

    raise ValueError(
        "ZENODO_SECRET_LINK is not in a recognized format. "
        "Provide a full link with token=... or the raw token."
    )

# ---- DOWNLOAD ----
os.makedirs(OUT_DIR, exist_ok=True)
dest = os.path.join(OUT_DIR, FILENAME)

if os.path.exists(dest) and os.path.getsize(dest) > 0:
    print(f"✅ Using cached file: {dest}")
else:
    token = get_zenodo_token_from_secret()
    url = f"https://zenodo.org/records/{RECORD_ID}/files/{FILENAME}?token={token}"

    max_retries = 6
    timeout_s = 300

    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Download attempt {attempt}/{max_retries}")
            with requests.get(url, stream=True, timeout=timeout_s) as r:
                if r.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {r.status_code}", response=r)
                r.raise_for_status()

                tmp = dest + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

                os.replace(tmp, dest)
                print(f"✅ Downloaded successfully: {dest}")
                break

        except Exception as e:
            # cleanup partial
            try:
                if os.path.exists(dest + ".part"):
                    os.remove(dest + ".part")
            except Exception:
                pass

            if attempt == max_retries:
                raise RuntimeError(f"❌ Download failed after {max_retries} attempts") from e

            wait = 2 * (2 ** (attempt - 1))
            print(f"⚠️ Error: {e}")
            print(f"⏳ Retrying in {wait}s…")
            time.sleep(wait)


⬇️ Download attempt 1/6
✅ Downloaded successfully: zenodo_downloads/lookup_amissenseXgnomAD.tsv


#**☁️ Upload a suitable .pdb**

*   If you followed **`part I`**  you can upload the file you Downloaded
*   Otherwise a file that meets **`requirements`**

We will verify that this file meets our requirements




In [ ]:
#@title ✅ Validate Selected PDB Input (UniProt ID + Select PDB) + Fetch & Unpack Zenodo tar.gz (after all tests pass)
#@markdown Mandatory inputs:
#@markdown - **UniProt ID**
#@markdown - **Click ▶️ Run** to open a **file picker in output cell** and select a **`.pdb`** file from your computer
#@markdown
#@markdown This cell:
#@markdown 1) Ensures local **gnomAD lookup TSV** exists (download from Zenodo if missing)
#@markdown 2) Validates **UniProt entry (reviewed Swiss-Prot, Homo sapiens)**
#@markdown 3) Extracts chain A from ATOM CA coords
#@markdown    - FAILS if **chain A has internal residue numbering gaps** (prints gapped sequence with '-' at gaps)
#@markdown    - Requires **100% identity** vs UniProt for residues present (allows missing N/C termini)
#@markdown 4) Confirms available **gnomAD allele frequency data** (lookup row exists)
#@markdown 5) If all pass: downloads the corresponding **.tar.gz** from the same Zenodo record and safely unpacks it

uniprot_id = "Q9Y2D9"  #@param {type:"string"}

import os, re, time
from pathlib import Path

import requests
import pandas as pd
import tarfile

# ==========================================================
# CONFIG
# ==========================================================
# UniProt REST
UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb/{accession}?format=json"

# Zenodo record + lookup file
RECORD_ID = "18163874"
LOOKUP_FILENAME = "lookup_amissenseXgnomAD.tsv"

# Storage locations
OUT_DIR = Path("zenodo_downloads")
LOOKUP_PATH = OUT_DIR / LOOKUP_FILENAME

TAR_DIR = OUT_DIR / "tars"
EXTRACT_DIR = OUT_DIR / "extracted"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TAR_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Globals for downstream steps
current_uniprot_id = None
current_sequence = None            # canonical: UniProt
current_sequence_uniprot = None
current_sequence_pdb = None        # chain A from ATOM coords (UNGAPPED)
current_pdb_path = None
has_gnomad_lookup = None
gnomad_lookup_path = None
current_tar_info = None

# ==========================================================
# TOKEN HELPERS (Colab Secrets OR env)
# ==========================================================
def get_zenodo_token_from_secret() -> str:
    """
    Reads ZENODO_SECRET_LINK from:
      1) Colab Secrets (google.colab.userdata)
      2) Environment variable

    Accepts either:
      - full link containing token=...
      - raw token string
    """
    secret = None
    try:
        from google.colab import userdata
        secret = userdata.get("ZENODO_SECRET_LINK")
    except Exception:
        secret = os.environ.get("ZENODO_SECRET_LINK")

    if not secret:
        raise ValueError("ZENODO_SECRET_LINK not found in Colab Secrets (or env).")

    secret = secret.strip()

    # If a full link (old approach)
    if "token=" in secret:
        return secret.split("token=", 1)[1].split("&", 1)[0]

    # If it's already a raw token
    if re.fullmatch(r"[A-Za-z0-9\-_\.]+", secret):
        return secret

    raise ValueError(
        "ZENODO_SECRET_LINK is not in a recognized format. "
        "Provide a full link with token=... or the raw token."
    )

# ==========================================================
# ZENODO DOWNLOAD (retry/backoff + atomic write)
# ==========================================================
def download_zenodo_file(record_id: str, filename: str, token: str, out_path: Path,
                         max_retries: int = 6, timeout_s: int = 300,
                         verbose: bool = False) -> Path:
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?token={token}"

    if out_path.exists() and out_path.stat().st_size > 0:
        if verbose:
            print(f"Using cached file: {out_path}")
        return out_path

    tmp = out_path.with_suffix(out_path.suffix + ".part")

    for attempt in range(1, max_retries + 1):
        try:
            if verbose:
                print(f"Downloading {filename} (attempt {attempt}/{max_retries})")

            with requests.get(url, stream=True, timeout=timeout_s) as r:
                if r.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {r.status_code}", response=r)
                r.raise_for_status()

                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)

            os.replace(tmp, out_path)
            return out_path

        except Exception:
            if tmp.exists():
                tmp.unlink()
            if attempt == max_retries:
                raise
            time.sleep(2 * (2 ** (attempt - 1)))

# ==========================================================
# ✅ LOOKUP TABLE ENSURER (this fixes the NameError)
# ==========================================================
def ensure_lookup_table() -> Path:
    """
    Ensure the gnomAD lookup TSV exists locally.
    - If already present and non-empty: return it
    - Else: download from Zenodo (requires token)
    Also performs a light schema check (required columns).
    """
    # If an earlier cell already created some path/variable, try to reuse it safely.
    for candidate_name in ("lookup_path", "gnomad_lookup_path", "LOOKUP_PATH"):
        if candidate_name in globals():
            val = globals()[candidate_name]
            try:
                p = Path(val)
                if p.exists() and p.stat().st_size > 0 and p.name.endswith(".tsv"):
                    # quick column check
                    df_head = pd.read_csv(p, sep="\t", nrows=5, dtype=str)
                    required = {"uniprot_id", "tar_file", "tsv_file"}
                    missing = required - set(df_head.columns)
                    if missing:
                        raise ValueError(
                            f"Lookup TSV exists but is missing required columns: {sorted(missing)}\n"
                            f"Found columns: {list(df_head.columns)}"
                        )
                    return p
            except Exception:
                pass  # fall back to standard LOOKUP_PATH below

    # Standard location
    out_path = LOOKUP_PATH

    if out_path.exists() and out_path.stat().st_size > 0:
        df_head = pd.read_csv(out_path, sep="\t", nrows=5, dtype=str)
        required = {"uniprot_id", "tar_file", "tsv_file"}
        missing = required - set(df_head.columns)
        if missing:
            raise ValueError(
                f"Lookup TSV exists but is missing required columns: {sorted(missing)}\n"
                f"Found columns: {list(df_head.columns)}"
            )
        return out_path

    # Need to download
    token = get_zenodo_token_from_secret()
    downloaded = download_zenodo_file(RECORD_ID, LOOKUP_FILENAME, token, out_path, verbose=True)

    # Validate schema after download
    df_head = pd.read_csv(downloaded, sep="\t", nrows=5, dtype=str)
    required = {"uniprot_id", "tar_file", "tsv_file"}
    missing = required - set(df_head.columns)
    if missing:
        raise ValueError(
            f"Downloaded lookup TSV is missing required columns: {sorted(missing)}\n"
            f"Found columns: {list(df_head.columns)}"
        )

    return downloaded

# ==========================================================
# SAFE TAR EXTRACTION
# ==========================================================
def _is_within_directory(base: Path, target: Path) -> bool:
    base = base.resolve()
    target = target.resolve()
    try:
        target.relative_to(base)
        return True
    except ValueError:
        return False

def safe_extract_tar_gz(tar_path: Path, out_dir: Path) -> None:
    """Safely extract tar.gz into out_dir (blocks path traversal, future-proof)."""
    out_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, "r:gz") as tf:
        # Explicit path traversal check
        for member in tf.getmembers():
            member_path = out_dir / member.name
            if not _is_within_directory(out_dir, member_path):
                raise RuntimeError(f"Unsafe tar member path: {member.name}")

        # Python 3.14+ safe extraction (fallback for older versions)
        try:
            tf.extractall(out_dir, filter="data")
        except TypeError:
            tf.extractall(out_dir)

def download_and_unpack_uniprot_tar(uniprot_id: str, lookup_tsv: Path) -> Path:
    df = pd.read_csv(lookup_tsv, sep="\t", dtype=str)

    hit = df[df["uniprot_id"].astype(str).str.strip() == uniprot_id]
    if hit.empty:
        raise ValueError(f"No gnomAD data for UniProt ID {uniprot_id}")

    row = hit.iloc[0]
    tar_file = row["tar_file"].strip()
    expected_tsv = row["tsv_file"].strip()

    token = get_zenodo_token_from_secret()

    tar_path = TAR_DIR / tar_file
    download_zenodo_file(RECORD_ID, tar_file, token, tar_path, verbose=False)

    stem = tar_file.replace(".tar.gz", "")
    out_dir = EXTRACT_DIR / stem

    if not (out_dir.exists() and any(out_dir.iterdir())):
        safe_extract_tar_gz(tar_path, out_dir)

    matches = list(out_dir.rglob(expected_tsv))
    if not matches:
        raise RuntimeError(f"Expected TSV not found: {expected_tsv}")

    return matches[0]

# ==========================================================
# PDB PICKER (Colab)
# ==========================================================
def select_pdb_via_picker() -> str:
    try:
        from google.colab import files
    except Exception:
        raise RuntimeError(
            "File picker requires Google Colab (google.colab.files). "
            "Please run this notebook in Colab."
        )

    print("📂 Please select a .pdb file from your machine...")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file selected. A .pdb file is required.")

    uploaded_name = next(iter(uploaded.keys()))
    if not uploaded_name.lower().endswith(".pdb"):
        raise ValueError(f"Selected file '{uploaded_name}' is not a .pdb")

    canonical = "input.pdb"

    # Overwrite canonical file
    if os.path.exists(canonical):
        os.remove(canonical)

    # Move uploaded file to canonical name
    os.replace(uploaded_name, canonical)

    # Optional cleanup of extra pdbs
    for fn in os.listdir("."):
        if fn != canonical and fn.lower().endswith(".pdb"):
            try:
                os.remove(fn)
            except Exception:
                pass

    return canonical

# ==========================================================
# UNIPROT HELPERS
# ==========================================================
def fetch_uniprot_entry(accession: str) -> dict:
    accession = accession.strip()
    if not accession:
        raise ValueError("UniProt ID is empty.")
    url = UNIPROT_ENTRY_URL.format(accession=accession)
    r = requests.get(url, timeout=60)
    if r.status_code == 404:
        raise ValueError(f"No UniProtKB entry found for '{accession}'.")
    if not r.ok:
        raise ValueError(f"UniProt API error (HTTP {r.status_code}).")
    return r.json()

def validate_entry(entry: dict) -> tuple[bool, str]:
    entry_type = entry.get("entryType", "")
    sci_name = entry.get("organism", {}).get("scientificName", "")

    is_reviewed = entry_type == "UniProtKB reviewed (Swiss-Prot)"
    is_human = "Homo sapiens" in sci_name

    lines = []
    lines.append("✓ UniProtKB reviewed (Swiss-Prot)" if is_reviewed else f"✗ entryType: {entry_type}")
    lines.append("✓ Organism: Homo sapiens (Human)" if is_human else f"✗ Organism: {sci_name}")
    return (is_reviewed and is_human), "\n".join(lines)

def extract_uniprot_sequence(entry: dict) -> str:
    seq = entry.get("sequence", {}).get("value")
    if not seq:
        raise ValueError("No sequence found in UniProt entry.")
    return seq.replace("\n", "").strip()

# ==========================================================
# PDB HELPERS (chain A from ATOM CA coords + gap detection)
# ==========================================================
_AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G","HIS":"H",
    "ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W",
    "TYR":"Y","VAL":"V","SEC":"U","PYL":"O",
}

def pdb_basic_sanity(pdb_text: str) -> None:
    if "ATOM" not in pdb_text:
        raise ValueError("PDB has no ATOM records; cannot extract sequence from coordinates.")

def _parse_atom_line(line: str):
    atom_name = line[12:16].strip()
    resname = line[17:20].strip()
    chain = line[21].strip() or "_"
    resseq_str = line[22:26].strip()
    icode = line[26].strip() or ""
    if not resseq_str:
        raise ValueError("Malformed PDB ATOM line: missing residue number.")
    return atom_name, resname, chain, int(resseq_str), icode

def extract_chainA_sequence_from_coords_with_gapinfo(pdb_text: str):
    """
    Extract chain A sequence from CA atoms.
    Also constructs a gapped-by-numbering sequence (inserting '-' for numbering jumps)
    so we can print where gaps are, then FAIL EARLY if gaps exist.
    """
    residues = []
    seen = set()
    has_chain_a = False

    for ln in pdb_text.splitlines():
        if not ln.startswith("ATOM"):
            continue
        atom_name, resname, chain, resseq, icode = _parse_atom_line(ln)
        if chain != "A":
            continue
        has_chain_a = True
        if atom_name != "CA":
            continue
        key = (resseq, icode)
        if key in seen:
            continue
        seen.add(key)
        residues.append((resseq, icode, resname))

    if not has_chain_a:
        raise ValueError("No chain A found in ATOM records.")
    if not residues:
        raise ValueError("Chain A found, but no CA atoms in ATOM records; cannot build residue sequence.")

    residues.sort(key=lambda x: (x[0], x[1]))

    seq_ungapped = "".join(_AA3_TO_1.get(resname.upper(), "X") for (_, _, resname) in residues)

    gapped = []
    gaps = []
    prev_resseq = residues[0][0]
    gapped.append(_AA3_TO_1.get(residues[0][2].upper(), "X"))

    for (resseq, icode, resname) in residues[1:]:
        jump = resseq - prev_resseq
        if jump > 1:
            gapped.extend(["-"] * (jump - 1))
            gaps.append((prev_resseq + 1, resseq - 1))
        gapped.append(_AA3_TO_1.get(resname.upper(), "X"))
        prev_resseq = resseq

    seq_gapped = "".join(gapped)
    return seq_ungapped, seq_gapped, gaps, residues

def require_chain_is_subsequence_of_uniprot(chain_seq: str, uniprot_seq: str) -> tuple[int, int]:
    """
    Require 100% identity for residues present, allowing missing residues only at N- and/or C-terminus.
    Equivalent to chain_seq being a contiguous substring of uniprot_seq.
    Returns (start_idx, end_idx) 0-based (end exclusive).
    """
    start = uniprot_seq.find(chain_seq)
    if start == -1:
        raise ValueError(
            "❌ Alignment/identity check failed: Chain A sequence is not an exact contiguous match within UniProt.\n"
            "This indicates at least one mismatch or an internal insertion/deletion relative to UniProt.\n"
        )
    return start, start + len(chain_seq)

# ==========================================================
# LOOKUP HELPERS
# ==========================================================
def lookup_row_for_uniprot(tsv_path: Path, uniprot_id: str) -> dict | None:
    df = pd.read_csv(tsv_path, sep="\t", dtype=str, low_memory=False)
    if "uniprot_id" not in df.columns:
        raise ValueError(f"Lookup TSV does not contain 'uniprot_id' column. Found: {list(df.columns)}")
    hit = df[df["uniprot_id"].astype(str).str.strip() == uniprot_id]
    if hit.empty:
        return None
    return hit.iloc[0].to_dict()

# ==========================================================
# RUN
# ==========================================================
try:
    uniprot_id = uniprot_id.strip()
    if not uniprot_id:
        raise ValueError("UniProt ID is required.")

    # 0) Ensure lookup table exists (download if needed)
    lookup_path = ensure_lookup_table()
    gnomad_lookup_path = str(lookup_path)

    # 1) Select PDB
    pdb_file = select_pdb_via_picker()
    current_pdb_path = pdb_file

    with open(pdb_file, "r", encoding="utf-8", errors="replace") as f:
        pdb_text = f.read()
    pdb_basic_sanity(pdb_text)

    # 2) Chain A extraction + FAIL EARLY on internal numbering gaps
    seq_chainA, seq_chainA_gapped, chainA_gaps, residues_chainA = extract_chainA_sequence_from_coords_with_gapinfo(pdb_text)
    current_sequence_pdb = seq_chainA

    print(f"✅ PDB loaded: {pdb_file}")
    print(f"✅ Chain A extracted from ATOM CA coords. Ungapped length: {len(seq_chainA)} aa")

    if chainA_gaps:
        gap_str = ", ".join([f"{a}-{b}" if a != b else f"{a}" for a, b in chainA_gaps])
        print(f"\n❌ FAIL: Chain A has INTERNAL residue numbering gap(s): {gap_str}")
        print("\nChain A sequence with '-' inserted for numbering gaps:")
        print(seq_chainA_gapped)
        raise ValueError("Stopping because PDB chain A has internal gaps.")
    else:
        print("✓ No internal residue numbering gaps detected in chain A.")
        print("\nChain A sequence (full, ungapped):")
        print(seq_chainA)
        print()

    # 3) UniProt validation
    entry = fetch_uniprot_entry(uniprot_id)
    ok, msg = validate_entry(entry)

    print(f"Checking UniProt entry for {uniprot_id}\n")
    print(msg)
    print()

    if not ok:
        raise ValueError("Stopping because UniProt criteria failed.")

    seq_uni = extract_uniprot_sequence(entry)
    current_sequence_uniprot = seq_uni
    current_uniprot_id = uniprot_id
    current_sequence = seq_uni  # canonical

    print(f"✅ Accepted. UniProt sequence length: {len(seq_uni)} aa")

    # 3b) End-trim tolerant 100% identity check
    start0, end0 = require_chain_is_subsequence_of_uniprot(seq_chainA, seq_uni)
    print("✅ Identity check: 100% match for residues present (allows N/C-terminal truncation).")
    print(f"Chain A maps to UniProt positions (1-based): {start0 + 1}-{end0}")
    print(f"N-terminal truncation vs UniProt: {start0} aa")
    print(f"C-terminal truncation vs UniProt: {len(seq_uni) - end0} aa")
    print()

    # 4) Confirm gnomAD allele frequency data exists (lookup row exists)
    row = lookup_row_for_uniprot(lookup_path, uniprot_id)
    has_gnomad_lookup = row is not None

    print(f"🗂️ gnomAD lookup table: {lookup_path}")
    print("gnomAD allele frequency availability for this UniProt ID:")
    if has_gnomad_lookup:
        print("✅ AVAILABLE")
        print(f"Found: tar_file={row.get('tar_file')}  tsv_file={row.get('tsv_file')}")
    else:
        print("❌ NOT FOUND")
        raise ValueError("Stopping because no gnomAD allele frequency data was found for this UniProt ID.")

    # 5) If all tests passed: download/unpack and print only success message
    gnomad_tsv_path = download_and_unpack_uniprot_tar(uniprot_id, lookup_path)
    print(f"✅ Retrieved gnomAD allele frequency data for {uniprot_id}")
    # (optional) print where it landed:
    # print(f"📄 TSV path: {gnomad_tsv_path}")

except Exception as e:
    print("❌ Error:", e)


📂 Please select a .pdb file from your machine...


Saving Q9Y2D9.pdb to Q9Y2D9.pdb
✅ PDB loaded: input.pdb
✅ Chain A extracted from ATOM CA coords. Ungapped length: 606 aa
✓ No internal residue numbering gaps detected in chain A.

Chain A sequence (full, ungapped):
MSHTASSCQELVENCAVHVAGMAQEDSRRGQVPSSFYHGANQELDLSTKVYKRESGSPYSVLVDTKMSKPHLHETEEQPYFRETRAVSDVHAVKEDRENSDDTEEEEEEVSYKREQIIVEVNLNNQTLNVSKGEKGVSSQSKETPVLKTSSEEEEEESEEEATDDSNDYGENEKQKKKEKIVEKVSVTQRRTRRAASVAAATTSPTPRTTRGRRKSVEPPKRKKRATKEPKAPVQKAKCEEKETLTCEKCPRVFNTRWYLEKHMNVTHRRMQICDKCGKKFVLESELSLHQQTDCEKNIQCVSCNKSFKKLWSLHEHIKIVHGYAEKKFSCEICEKKFYTMAHVRKHMVAHTKDMPFTCETCGKSFKRSMSLKVHSLQHSGEKPFRCENCDERFQYKYQLRSHMSIHIGHKQFMCQWCGKDFNMKQYFDEHMKTHTGEKPFICEICGKSFTSRPNMKRHRRTHTGEKPYPCDVCGQRFRFSNMLKAHKEKCFRVTSPVNVPPAVQIPLTTSPATPVPSVVNTATTPTPPINMNPVSTLPPRPIPHPFSHLHIHPHPHHPHHLPIPPVPHLPPPPALFKSEPLNHRGQSEDNFLRHLAEKNSSAQHH

Checking UniProt entry for Q9Y2D9

✓ UniProtKB reviewed (Swiss-Prot)
✓ Organism: Homo sapiens (Human)

✅ Accepted. UniProt sequence length: 606 aa
✅ Identity check: 100% match f

#**🏗️ Build model with allele frequency and alternate sidechains in .pdb 🧭🗺️📍**

In [ ]:
#@title Run gnomAD structure mapping (≈30 min for ~500 aa) ☕🍵
#@markdown ---
#@markdown ### Mapping mode
#@markdown This step maps **gnomAD missense variation** onto the protein structure.
#@markdown
#@markdown **Allele frequency**
#@markdown - Uses *global* allele frequency across all individuals
#@markdown
#@markdown **Ancestry frequency**
#@markdown - Uses *population- or sex-specific* allele frequencies
#@markdown
#@markdown ---
#@markdown ### Allowed gnomAD categories
#@markdown **Population prefixes**
#@markdown - `nfe` — Non-Finnish European
#@markdown - `fin` — Finnish
#@markdown - `mid` — Middle Eastern
#@markdown - `remaining` — Remaining / unassigned populations
#@markdown - `ami` — Amish
#@markdown - `asj` — Ashkenazi Jewish
#@markdown - `afr` — African / African American
#@markdown - `eas` — East Asian
#@markdown - `sas` — South Asian
#@markdown - `amr` — Admixed American
#@markdown
#@markdown **Biological sex prefixes**
#@markdown - `XX` — Biological females
#@markdown - `XY` — Biological males
#@markdown
#@markdown ⚠️ **Important constraint**
#@markdown - A maximum of **7 categories** can be used at once
#@markdown - Categories are mapped to chains **T, U, V, W, X, Y, Z**
#@markdown - The run will **fail immediately** if more than 7 are provided
#@markdown ---

import os, sys, subprocess, time
from pathlib import Path

# -------------------------
# Parameters
# -------------------------
mapping = "ancestry frequency"  #@param ["allele frequency", "ancestry frequency"]
CATEGORIES = "afr,nfe,sas,eas,fin"  #@param {type:"string"}

# -------------------------
# Resolve inputs from previous cells
# -------------------------
def _as_path(x):
    if x is None:
        return None
    return x if isinstance(x, Path) else Path(str(x))

tsv_path = _as_path(globals().get("gnomad_tsv_path"))
pdb_path = _as_path(globals().get("current_pdb_path"))

if tsv_path is None or pdb_path is None:
    raise RuntimeError(
        "Missing inputs.\n"
        "Please run the previous validation/download cell (Part II) before this step."
    )

base_out = Path("morph_outputs").resolve()

if mapping == "allele frequency":
    out_dir = base_out / "allele_frequency"
else:
    out_dir = base_out / "ancestry_frequency"

out_dir.mkdir(parents=True, exist_ok=True)


# -------------------------
# Enforce mode-specific logic
# -------------------------
if mapping == "allele frequency":
    # Hard-disable categories in allele-frequency mode
    CATEGORIES = None
    SCRIPT = Path(
        "pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_af_2_structure.py"
    ).resolve()

elif mapping == "ancestry frequency":
    SCRIPT = Path(
        "pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_anc_2_structure.py"
    ).resolve()

    cats = [c.strip() for c in CATEGORIES.split(",") if c.strip()]

    if not cats:
        raise ValueError(
            "Ancestry frequency mapping requires at least one gnomAD category."
        )

    if len(cats) > 7:
        raise ValueError(
            f"You specified {len(cats)} categories, but the maximum allowed is 7.\n"
            "This limit exists because categories are mapped to chains T–Z.\n"
            "Please reduce the list and try again."
        )

else:
    raise ValueError(f"Unknown mapping mode: {mapping}")

# -------------------------
# Sanity checks
# -------------------------
def _must_exist(p: Path, label: str):
    if not p.exists():
        raise FileNotFoundError(f"{label} not found: {p}")

_must_exist(SCRIPT, "SCRIPT")
_must_exist(tsv_path, "TSV")
_must_exist(pdb_path, "PDB")

# -------------------------
# Ensure subprocess can import pyMUT
# -------------------------
env = os.environ.copy()
pymut_dir = str(Path("pyMUT").resolve())
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = ":".join(dict.fromkeys([pymut_dir] + existing.split(":")))

# -------------------------
# Build command
# -------------------------
if mapping == "allele frequency":
    cmd = [sys.executable, "-u", str(SCRIPT), str(tsv_path), str(pdb_path), str(out_dir)]
else:
    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),
        str(tsv_path),
        str(pdb_path),
        str(out_dir),
        CATEGORIES,
    ]

print("Running:", " ".join(cmd))

# -------------------------
# Run with live output
# -------------------------
TIMEOUT_SECONDS = 3600
start = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in iter(proc.stdout.readline, ""):
        print(line, end="")
        if time.time() - start > TIMEOUT_SECONDS:
            proc.kill()
            raise TimeoutError("Process exceeded time limit and was terminated.")
    proc.stdout.close()
    rc = proc.wait()
finally:
    try:
        proc.kill()
    except Exception:
        pass

if rc != 0:
    raise RuntimeError(f"Script failed with return code {rc}")

print("\n✅ gnomAD mapping completed successfully.")


Running: /usr/bin/python3 -u /content/pyMUT/src/2-mapping/map_2_structure/pymut_gnomad_anc_2_structure.py zenodo_downloads/extracted/amissenseXgnomAD_part0096of0099/Q9Y2D9_amissenseXgnomAD.tsv input.pdb /content/morph_outputs/ancestry_frequency afr,nfe,sas,eas,fin
[12:51:30] 1) TSV -> per-category top variant per position (t+0:00:00.000015)
[12:51:30]  - completed TSV processing (categories: 5) (t+0:00:00.212671)
[12:51:30] 2) WT residue map from chain A (original PDB) (t+0:00:00.212785)
[12:51:30] 3) Normalize original PDB (free reserved chain ids + normalize residue names) (t+0:00:00.256000)
[12:51:30] 4) Create + mutate category chains using pyMUT (single-chain temp PDBs) (t+0:00:00.325632)
[12:51:30] --- Category 'afr' -> Chain 'V' --- (t+0:00:00.325741)
 - pyMUT phase1 V: 50/94 mutations applied
 - pyMUT phase1 V: 94/94 mutations applied
 - pyMUT phase2 V: clashing sites=69, remutated=69/69 (hard<3.00Å; neighbor<8.0Å)
[12:55:46]  - V: mutated 94 sites (pyMUT=first; fallback=best; 

In [ ]:
#@title Zip and download mapping outputs
#@markdown This will package and download the outputs produced by the selected mapping mode.

from pathlib import Path
import shutil

# Base output directory (must match the run cell)
base_out = Path("morph_outputs").resolve()

# Select subdirectory based on mapping mode
if mapping == "allele frequency":
    out_dir = base_out / "allele_frequency"
    zip_name = "morph_allele_frequency_outputs"
elif mapping == "ancestry frequency":
    out_dir = base_out / "ancestry_frequency"
    zip_name = "morph_ancestry_frequency_outputs"
else:
    raise ValueError(f"Unknown mapping mode: {mapping}")

# Sanity check
if not out_dir.exists():
    raise FileNotFoundError(
        f"Expected output directory not found:\n  {out_dir}\n"
        "Please run the mapping cell first."
    )

zip_path = out_dir.parent / f"{zip_name}.zip"

# Remove old archive if present
if zip_path.exists():
    zip_path.unlink()

# Create archive
shutil.make_archive(
    base_name=str(zip_path.with_suffix("")),
    format="zip",
    root_dir=out_dir.parent,
    base_dir=out_dir.name,
)

print(f"Created archive: {zip_path}")

# Download (Colab)
from google.colab import files
files.download(str(zip_path))


NameError: name 'mapping' is not defined